# Automaton — free Qwen 3B endpoint on Kaggle

This notebook runs **Qwen2.5-3B-Instruct** on a Kaggle GPU and exposes the OpenAI-compatible endpoint Automaton expects.

1. In Kaggle, create a notebook and enable **GPU T4** and **Internet** in notebook settings.
2. Import this `.ipynb`, then run every cell.
3. Copy the final `LLM_BASE_URL` and `LLM_API_KEY` values into the environment of your Automaton web deployment.

**Limits:** Kaggle sessions are temporary and can stop. This is suitable for testing and early launches, not permanent 24/7 inference. The endpoint uses a random bearer token, but its temporary tunnel is public—never remove authentication.


In [ ]:
!pip install -q "transformers>=4.45,<5" "accelerate>=0.34" "peft>=0.14,<1" fastapi uvicorn pydantic
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/cloudflared
!chmod +x /kaggle/working/cloudflared


In [ ]:
import os, secrets, threading, time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
# Optional promoted adapter uploaded as a Kaggle Dataset; leave blank for base model.
ADAPTER_PATH = ""
AUTOMATON_TOKEN = secrets.token_urlsafe(32)

print("Loading", MODEL_ID)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)
if ADAPTER_PATH:
    if not os.path.isdir(ADAPTER_PATH):
        raise FileNotFoundError(f"Adapter path not found: {ADAPTER_PATH}")
    model = PeftModel.from_pretrained(model, ADAPTER_PATH)
    print("Loaded promoted Automaton adapter:", ADAPTER_PATH)
print("Model ready on", model.device)


In [ ]:
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
from typing import Any
import uvicorn

api = FastAPI()

class ChatRequest(BaseModel):
    model: str | None = None
    messages: list[dict[str, Any]]
    temperature: float = 0.6
    max_tokens: int = 1400
    response_format: dict | None = None

@api.get("/health")
def health():
    return {"ok": True, "model": MODEL_ID}

@api.post("/v1/chat/completions")
def chat(body: ChatRequest, authorization: str = Header(default="")):
    if authorization != f"Bearer {AUTOMATON_TOKEN}":
        raise HTTPException(401, "Invalid bearer token")
    text = tokenizer.apply_chat_template(
        body.messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=min(body.max_tokens, 1800),
            do_sample=True,
            temperature=max(0.1, min(body.temperature, 1.2)),
            top_p=0.9,
            repetition_penalty=1.05,
        )
    output = tokenizer.decode(
        generated[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
    )
    return {
        "id": "qwen-kaggle",
        "object": "chat.completion",
        "choices": [{"index": 0, "message": {"role": "assistant", "content": output}, "finish_reason": "stop"}],
        "usage": {},
    }

def serve():
    uvicorn.run(api, host="0.0.0.0", port=8000, log_level="warning")

threading.Thread(target=serve, daemon=True).start()
time.sleep(3)
print("Local inference API started")


In [ ]:
import re, subprocess

proc = subprocess.Popen(
    ["/kaggle/working/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
public_url = None
deadline = time.time() + 60
while time.time() < deadline:
    line = proc.stdout.readline()
    print(line, end="")
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break
if not public_url:
    raise RuntimeError("Tunnel did not start. Re-run this cell.")

print("\nCOPY THESE TWO VALUES TO YOUR AUTOMATON HOST:")
print(f"LLM_BASE_URL={public_url}/v1")
print(f"LLM_API_KEY={AUTOMATON_TOKEN}")
print(f"LLM_MODEL={MODEL_ID}")


## Keep the serving process open

The final cell blocks with near-zero CPU so FastAPI and the tunnel can serve real inference requests while Kaggle permits the session. It does **not** fake GPU activity, self-ping, or bypass Kaggle quotas. Kaggle can still stop the notebook at an idle, weekly, or maximum-session limit. If that happens, Automaton uses its deterministic fulfillment fallback until you start a permitted session and update `LLM_BASE_URL`.


In [ ]:
import threading
print("Qwen endpoint is active while this permitted Kaggle session remains available.")
print("This zero-CPU wait does not and cannot override Kaggle session limits.")
threading.Event().wait()
